In [ ]:
# Installing needed package (triton)
! pip install triton

In [17]:
# Importing needed libraries
import torch
import torch.nn.functional as F
import triton
import triton.language as tl

In [18]:
# Global Variables
DEVICE = "cuda"
C_OUT = 64
C_IN = 3
H = 1024
W = 1024
FH = 3
FW = 3

In [19]:
# Making torch tensors with specified initialization
# I[c, x, y] = c * (x + y)
c_i = torch.arange(C_IN, device=DEVICE).view(1, C_IN, 1, 1)
y_i = torch.arange(H, device=DEVICE).view(1, 1, H, 1)
x_i = torch.arange(W, device=DEVICE).view(1, 1, 1, W)
tensor_I = (c_i * (x_i + y_i)).to(torch.float32)

# F[k, c, i, j] = (c + k) * (i + j)
k_f = torch.arange(C_OUT, device=DEVICE).view(C_OUT, 1, 1, 1)
c_f = torch.arange(C_IN, device=DEVICE).view(1, C_IN, 1, 1)
i_f = torch.arange(FH, device=DEVICE).view(1, 1, FH, 1)
j_f = torch.arange(FW, device=DEVICE).view(1, 1, 1, FW)
tensor_F = ((c_f + k_f) * (i_f + j_f)).to(torch.float32)

print(f"tensor_I shape: {tensor_I.shape}")
print(f"tensor_F shape: {tensor_F.shape}")

tensor_I shape: torch.Size([1, 3, 1024, 1024])
tensor_F shape: torch.Size([64, 3, 3, 3])


In [20]:
# This is the result from Convolutional Layer provided by Torch
# Use this for correctness check
golden_out = F.conv2d(tensor_I, tensor_F, padding=1)
print(golden_out.shape) # (1, C_OUT, OUT_H, OUT_W)

torch.Size([1, 64, 1024, 1024])


In [25]:
@triton.jit
def my_triton_kernel(
    input_ptr, weight_ptr, output_ptr,
    BATCH, C_IN, H, W, 
    C_OUT, FH, FW,
    stride_in_b, stride_in_c, stride_in_h, stride_in_w,
    stride_w_k, stride_w_c, stride_w_h, stride_w_w,
    stride_out_b, stride_out_k, stride_out_h, stride_out_w,
    BLOCK_SIZE_H: tl.constexpr, BLOCK_SIZE_W: tl.constexpr
):
    # Standard convolution kernel logic with padding=1, stride=1
    pid_h = tl.program_id(0) # spatial tile row
    pid_w = tl.program_id(1) # spatial tile col
    pid_k = tl.program_id(2) # output channel index

    # Offsets for current spatial tile within output feature map
    offsets_h = pid_h * BLOCK_SIZE_H + tl.arange(0, BLOCK_SIZE_H)
    offsets_w = pid_w * BLOCK_SIZE_W + tl.arange(0, BLOCK_SIZE_W)

    # Intermediate accumulator
    acc = tl.zeros((BLOCK_SIZE_H, BLOCK_SIZE_W), dtype=tl.float32)

    # Channel loop
    for c in range(0, C_IN):
        # Spatial filter loop
        for fh in range(0, FH):
            for fw in range(0, FW):
                # Load weight scalar for current (k, c, fh, fw)
                curr_w_ptr = (weight_ptr + pid_k * stride_w_k + c * stride_w_c + 
                              fh * stride_w_h + fw * stride_w_w)
                weight = tl.load(curr_w_ptr)

                # Calculate input indices: H_in = H_out + fh - 1, W_in = W_out + fw - 1
                # This assumes padding=1
                h_in = offsets_h[:, None] + fh - 1
                w_in = offsets_w[None, :] + fw - 1

                # Handle boundaries
                mask = (h_in >= 0) & (h_in < H) & (w_in >= 0) & (w_in < W)

                # Pointer math for tiled input load
                curr_in_ptr = (input_ptr + c * stride_in_c + 
                               h_in * stride_in_h + w_in * stride_in_w)
                
                # Direct tiled load from Global Memory
                input_val = tl.load(curr_in_ptr, mask=mask, other=0.0)

                # Update tile accumulator
                acc += weight * input_val

    # Final pointer offsets for output feature map write-back
    out_offsets = (output_ptr + pid_k * stride_out_k + 
                   offsets_h[:, None] * stride_out_h + offsets_w[None, :] * stride_out_w)
    
    # Store the finished BLOCK x BLOCK tile
    out_mask = (offsets_h[:, None] < H) & (offsets_w[None, :] < W)
    tl.store(out_offsets, acc, mask=out_mask)

def my_conv2d(input, kernel):
    BATCH, C_IN, H, W = input.shape
    C_OUT, _, FH, FW = kernel.shape
    
    # Assuming padding=1, stride=1, output matches input spatial size
    H_out, W_out = H, W
    
    DEVICE = input.device
    output = torch.zeros((BATCH, C_OUT, H_out, W_out), device=DEVICE, dtype=torch.float32)
    
    # Block size configuration
    BLOCK_SIZE_H = 32
    BLOCK_SIZE_W = 32
    
    grid = (
        triton.cdiv(H_out, BLOCK_SIZE_H),
        triton.cdiv(W_out, BLOCK_SIZE_W),
        C_OUT
    )
    
    # Kernel runner
    def run_conv():
        my_triton_kernel[grid](
            input, kernel, output,
            BATCH, C_IN, H, W,
            C_OUT, FH, FW,
            *input.stride(),
            *kernel.stride(),
            *output.stride(),
            BLOCK_SIZE_H=BLOCK_SIZE_H,
            BLOCK_SIZE_W=BLOCK_SIZE_W,
        )
    
    execution_time = triton.testing.do_bench(run_conv)

    return output, execution_time

In [26]:
# Testing
# Comparing the result from my_conv2d and Conv from torch
try:
    my_output, execution_time = my_conv2d(tensor_I, tensor_F)
    torch.testing.assert_close(golden_out, my_output) # Assert statement should be passed
    # Printing the execution time
    print(f"Average execution time per kernel: {execution_time:.4f} ms")
except Exception as e:
    print(f"Kernel execution failed: {e}")

Average execution time per kernel: 17.0869 ms
